# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.limit(10).display()

## Customer ID Cleanup

- Performing conditional Prefix Removal step.
- Checking if the id starts with 'NAS'. We strip that and keep the remaining data.
- Mostly common when dealing with legacy data.

In [0]:
df = df.withColumn(
    'cid',
    F.when(col('cid').startswith('NAS'),
            F.substring(col("CID"), 4, F.length(col("CID"))))
    .otherwise(col("CID"))
    
    )


## Birthdate Validation

- Checking if the birthdate is not more that current date.

In [0]:
df = df.withColumn(
    'bdate',
    F.when(col("BDATE") > F.current_date(), None)
    .otherwise(col("BDATE"))
)

## Gender Normalisation

In [0]:
df = df.withColumn(
    'gen',
    F.when(F.upper(col('GEN')).isin('F','FEMALE'),'Female')
    .when(F.upper(col("GEN")).isin('M','MALE'), 'MALE')
    .otherwise('N/A')
)

## Renaming Columns 

In [0]:
RENAME_MAP = {
    'cid': 'customer_number',
    'bdate': 'birth_date',
    'gen': 'gender'

}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

## Sanity Checks

In [0]:
df.limit(10).display()

# Writing in Silver Table

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customer')

# Sanity check for silver table

In [0]:
%sql
select * from workspace.silver.erp_customer